# Diabetes prediction

## Diabetes Prediction - Project Overview

This project compares two diabetes datasets that differ substantially in what kind of signal they offer a model.

The Diabetes prediction dataset (~100k rows) is a clinically-oriented dataset: alongside demographic and lifestyle fields, it includes lab-derived measurements - `HbA1c_level` and `blood_glucose_level` - that are themselves part of the official diagnostic criteria for diabetes.

The Diabetes Health Indicators Dataset, derived from the CDC's BRFSS 2015 survey (~250k+ rows), contains no lab measurements at all - only self-reported lifestyle, demographic, and comorbidity indicators (blood pressure, cholesterol, general health, physical activity, income, education, etc.), making it a harder and arguably more realistic task: predicting diabetes risk purely from indirect, survey-based population health data rather than direct clinical readings.

Training both datasets side by side makes this contrast explicit: the first shows what's achievable when direct clinical markers are available, while the second shows what's achievable when they aren't - which is the more relevant scenario for large-scale, low-cost screening.

## Dataset Overview - Diabetes prediction dataset (Clinical)

Some attributes like age and gender are self-explanatory. However, attributes like HbA1c_level, blood_glucose_level, and even bmi require domain knowledge, which is why we go through some explanations below.

**BMI (Body Mass Index)**

bmi: Body Mass Index. A generally useful metric, calculated as weight (kg) / height (m)$^2$. However, it does not account for age, sex, muscle mass, or body fat percentage.

Standard World Health Organization classification:
- Underweight: < 18.5
- Normal weight: 18.5 – 24.9
- Overweight: 25.0 – 29.9
- Obese: 30.0+

**HbA1c_level (Glycated Hemoglobin)**

Reflects the average blood sugar level over the past 2–3 months, measured as a percentage. Unlike a single glucose reading, it is not affected by short-term fluctuations (e.g. what the person ate that morning), which makes it a more stable diagnostic indicator.

Clinical thresholds:
- Normal: < 5.7%
- Prediabetes: 5.7% – 6.4%
- Diabetes: $\geq$ 6.5%

**blood_glucose_level**

A single snapshot measurement of sugar concentration in the blood at the time of testing, expressed in mg/dL (milligrams of glucose per deciliter of blood) - unlike HbA1c which reflects a longer-term average. Depending on whether the reading was fasting or random, the diagnostic thresholds differ:

Fasting glucose:
- Normal: < 100 mg/dL
- Prediabetes: 100 – 125 mg/dL
- Diabetes: $\geq$ 126 mg/dL

Random (non-fasting) glucose:
- Diabetes: $ \geq $ 200 mg/dL

> Note: this dataset does not specify whether readings are fasting or random, which is a limitation worth mentioning in the analysis.

**smoking_history**

Categorical variable with 6 possible values: `never`, `former`, `current`, 
`not current`, `ever`, and `No Info`.

The `No Info` category is not a smoking status - it indicates missing data (the respondent's smoking history was not recorded). Treating it as its own category, rather than dropping it, avoids losing rows, but it should not be interpreted as "never smoked."

The overlap between `former`, `not current`, and `ever` is somewhat ambiguous in the source data and worth noting as a limitation - these may need to be consolidated during preprocessing.

**hypertension** and **heart_disease**

Both are binary indicators (0 = No, 1 = Yes) representing whether the respondent has been previously diagnosed with high blood pressure or heart disease, respectively. These are included as established comorbidity risk factors for type 2 diabetes - hypertension and cardiovascular disease frequently co-occur with diabetes due to shared risk factors such as obesity and metabolic syndrome.

**diabetes** (target variable)

Binary label (0 = No diabetes, 1 = Diabetes). Note the overlap with the clinical thresholds discussed above - since HbA1c $ \geq $ 6.5% and fasting glucose $ \geq $ 126 mg/dL are themselves diagnostic criteria for diabetes, these two features are likely to be extremely strong (possibly near-deterministic) predictors of the target. This should be flagged explicitly, as models may achieve very high accuracy simply by learning the clinical cutoff rather than genuine risk patterns from lifestyle/demographic features.

## Dataset Overview - Diabetes Health Indicators Dataset (BRFSS Survey)

This dataset is derived from the CDC's 2015 Behavioral Risk Factor Surveillance System (BRFSS) survey. Unlike the smaller diabetes prediction dataset, it contains no lab measurements - every feature is either self-reported or derived from a survey question, so domain knowledge is needed to interpret several of them correctly.

**general_health**
Self-reported overall health on a 5-point ordinal scale (1 = Excellent to 5 = Poor). It is a subjective measure, but consistently one of the strongest predictors of diabetes status across BRFSS-based studies, likely because it implicitly captures a wide range of unmeasured health conditions.

**high_blood_pressure** and **high_cholesterol**
Binary indicators (0 = No, 1 = Yes) for a prior diagnosis of hypertension or high cholesterol, respectively. Both are established comorbidities of type 2 diabetes, frequently co-occurring due to shared risk factors such as obesity and metabolic syndrome.

**chol_checked_recently**
Binary indicator of whether the respondent had their cholesterol checked within the last 5 years. This is not a risk factor in the clinical sense - it is a healthcare-engagement proxy. People who are never screened are also less likely to be screened and diagnosed for diabetes, so this feature can pick up a detection-bias signal rather than a genuine physiological one. This should be flagged explicitly, since a model may lean on it for reasons unrelated to real diabetes risk.

**bmi**
Body Mass Index, calculated as weight (kg) / height (m)$^2$. Same World Health Organization classification as in the other dataset:
- Underweight: < 18.5
- Normal weight: 18.5 – 24.9
- Overweight: 25.0 – 29.9
- Obese: 30.0+

**physical_health_bad_days** and **mental_health_bad_days**
Self-reported number of days (0–30) in the past month during which the respondent's physical or mental health was "not good." These are continuous proxies for general health burden rather than diabetes-specific indicators.

**difficulty_walking**
Binary indicator (0 = No, 1 = Yes) for serious difficulty walking or climbing stairs - a proxy for mobility limitations, which can be both a cause (reduced physical activity) and a consequence (neuropathy, obesity) of diabetes.

**had_stroke** and **heart_disease_or_attack**
Binary indicators for a prior diagnosis of stroke or coronary heart disease/myocardial infarction. Like hypertension and high cholesterol, these are established diabetes comorbidities rather than direct predictors.

**physically_active**, **consumes_fruit_daily**, **consumes_veggies_daily**, **heavy_alcohol_consumption**, **smoked_at_least_100_cigarettes**
Binary lifestyle indicators self-reported by the respondent (e.g. "ever smoked at least 100 cigarettes" is the standard BRFSS proxy for "ever a smoker"). These carry real signal but are individually weak, indirect predictors compared to the comorbidity and general-health features above.

**skipped_doctor_due_to_cost**
Binary indicator of whether cost prevented the respondent from seeing a doctor when needed in the past year - a healthcare-access proxy that, similarly to `chol_checked_recently`, can reflect socioeconomic and detection-bias effects rather than physiological risk.

**income_level** and **education_level**
Ordinal categorical variables (grouped income brackets and education levels). Both are socioeconomic proxies, correlated with diet, healthcare access, and diagnosis rates rather than direct physiological risk factors.

**age** and **sex**
`age` is reported in BRFSS as a grouped ordinal category (13 age brackets) rather than an exact value, which should be kept in mind when interpreting model coefficients or feature importances. `sex` is binary (0/1).

**diabetes** (target variable)
Binary label (0 = No diabetes, 1 = Prediabetes, 2 = Diabetes), with a substantial class imbalance (~8.8% positive). Because none of the features here are direct lab measurements, no single feature is expected to be near-deterministic the way `HbA1c_level` or `blood_glucose_level` are in the other dataset - predictive performance instead comes from combining many weak, indirect signals, which is the main reason overall metrics (ROC-AUC, PR-AUC) are noticeably lower here than on the lab-based dataset.

## Conclusion

## References

### Datasets

[Diabetes Health Indicators Dataset](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset)

[Behavioral Risk Factor Surveillance System](https://www.kaggle.com/datasets/cdc/behavioral-risk-factor-surveillance-system)

[Diabetes prediction dataset](https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset)

### Academic References

[Diabetes](https://www.mayoclinic.org/diseases-conditions/diabetes/symptoms-causes/syc-20371444)

[Prediabetes](https://www.mayoclinic.org/diseases-conditions/prediabetes/symptoms-causes/syc-20355278)